# Clasificación de Texto Tradicional vs. Aprendizaje Profundo

**Materiales desarrollados por Matías Barreto, 2026**

**Tecnicatura Superior en Ciencias de Datos e IA, IFTS24**
* **Nomenclatura Oficial:** Procesamiento de Lenguaje Natural
* **Nombre de Trabajo:** Laboratorio de PLN: Analítica, Textos y Cultura

---

## Objetivo
Introducir la clasificación de texto tradicional utilizando la vectorización por TF-IDF y modelos lineales de Scikit-Learn como baseline de comparación frente al aprendizaje profundo.

## Resultados de aprendizaje
Al final de este notebook vas a poder:
1. Preprocesar texto y extraer características numéricas usando TF-IDF.
2. Entrenar y evaluar un modelo de clasificación lineal (Regresión Logística) con Scikit-Learn.
3. Interpretar la matriz de confusión y las métricas de precisión, exhaustividad y F1-score.



## Terminología clave (Microglosario)

* **✦ Clasificación de Texto:** Tarea de PLN que consiste en asignar automáticamente etiquetas categóricas a documentos escritos según su contenido semántico.
* **✦ TF-IDF (Term Frequency - Inverse Document Frequency):** Medida estadística que evalúa la relevancia de una palabra en un documento en relación con un corpus completo, sopesando términos comunes y raros.
* **✦ Regresión Logística:** Algoritmo lineal de aprendizaje supervisado utilizado para tareas de clasificación que estima probabilidades de pertenencia a clases.



---

## 1. Instalación de Dependencias

Instalamos Faker para generar un dataset sintético pero realista en español.

In [ ]:
# Instalación de librerías necesarias
# Faker: Para generar datos sintéticos realistas
%pip install -q Faker

print("OK Dependencias instaladas correctamente.")

---

## 2. Importación de Librerías

Importamos las herramientas necesarias: scikit-learn para ML, pandas para datos, y datasets para cargar el corpus.

In [ ]:
# Librerías para manipulación de datos
import pandas as pd
import numpy as np
import random

# Vectorización de texto de scikit-learn
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# División de datos y modelo
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

# Métricas de evaluación
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Para generar datos sintéticos realistas
from faker import Faker

print("OK Librerías importadas correctamente.")

---

## 3. Generación del Dataset

Vamos a crear un dataset **sintético pero realista** de reseñas de productos en español argentino usando técnicas profesionales de generación de datos.

Este dataset incluye:
-  **Variedad lingüística**: Español rioplatense con expresiones locales
-  **Casos complejos**: Ironía, sarcasmo, negaciones
-  **Realismo**: Combinaciones naturales de adjetivos y contextos
-  **Balance**: Distribución equilibrada de sentimientos

### ¿Por qué sintético?

Los datasets públicos de reviews en español tienen limitaciones (deprecated, poco balanceados, o en otros dialectos). Un dataset sintético bien diseñado nos permite:

1. Controlar el balance de clases
2. Incluir casos pedagógicamente útiles (ironía, negaciones)
3. Usar español argentino auténtico
4. Ajustar el tamaño según necesidad (clase vs TP)

In [ ]:
# Configuramos seeds para reproducibilidad
fake = Faker('es_ES')
Faker.seed(42)
random.seed(42)
np.random.seed(42)

# ============================================================================
# TEMPLATES DE RESEÑAS POSITIVAS SIMPLES
# ============================================================================

# Adjetivos positivos (español rioplatense)
adjetivos_positivos = [
    "excelente", "genial", "buenísimo", "espectacular", "increíble",
    "perfecto", "maravilloso", "bárbaro", "copado", "grosso",
    "de primera", "impecable", "fantástico", "hermoso", "divino"
]

# Verbos positivos
verbos_positivos = [
    "me encantó", "me fascina", "lo recomiendo", "superó mis expectativas",
    "cumple perfectamente", "funciona de diez", "vale la pena",
    "estoy re contento", "no me arrepiento", "la rompe"
]

# Contextos positivos
contextos_positivos = [
    "Llegó antes de tiempo y en perfecto estado.",
    "La calidad es superior a lo que esperaba.",
    "El vendedor fue muy atento y respondió todas mis dudas.",
    "Por este precio, es una ganga total.",
    "Mis amigos quedaron fascinados cuando lo vieron.",
    "Ya es la segunda vez que compro y sigue siendo excelente.",
    "Lo uso todos los días y sigue como nuevo.",
    "Mi familia está encantada con la compra.",
]

templates_positivos = [
    "{adj} producto, {verbo}. {contexto}",
    "{verbo}, {adj} compra. {contexto}",
    "El producto es {adj}. {contexto} {verbo}.",
    "{contexto} Realmente {adj}, {verbo}.",
    "{adj} en todo sentido. {verbo}. {contexto}",
]

# ============================================================================
# TEMPLATES DE RESEÑAS NEGATIVAS SIMPLES
# ============================================================================

# Adjetivos negativos (español rioplatense)
adjetivos_negativos = [
    "horrible", "malísimo", "pésimo", "terrible", "espantoso",
    "desastroso", "deplorable", "trucho", "berreta", "choto",
    "un desastre", "una porquería", "un fiasco", "decepcionante", "lamentable"
]

# Verbos negativos
verbos_negativos = [
    "no lo recomiendo", "me arrepiento de comprarlo", "pérdida de plata",
    "tuve que devolverlo", "no funciona", "se rompió enseguida",
    "no vale la pena", "es una estafa", "no cumple lo prometido",
    "quedé re decepcionado"
]

# Contextos negativos
contextos_negativos = [
    "Se rompió a los pocos días de uso.",
    "La calidad es muy inferior a la descripción.",
    "El vendedor no responde los mensajes.",
    "Tardó más de un mes en llegar.",
    "Llegó todo golpeado y con partes faltantes.",
    "No se parece en nada a las fotos.",
    "Hace ruidos extraños y se sobrecalienta.",
    "El material es plástico barato de mala calidad.",
]

templates_negativos = [
    "{adj} producto, {verbo}. {contexto}",
    "{verbo}, {adj} experiencia. {contexto}",
    "El producto es {adj}. {contexto} {verbo}.",
    "{contexto} Realmente {adj}, {verbo}.",
    "{adj} en todo sentido. {verbo}. {contexto}",
]

# ============================================================================
# CASOS DIFÍCILES: IRONÍA Y SARCASMO (Sin señales obvias)
# ============================================================================

# IRONÍA POSITIVA: Empieza mal pero termina bien (señal confusa al principio)
casos_ironia_positiva = [
    "Pensé que iba a ser horrible, pero me equivoqué completamente.",
    "Las primeras impresiones eran malas, terminó siendo muy útil.",
    "No confiaba en este producto, ahora lo uso todos los días.",
    "Dudaba mucho, pero resultó ser mejor que productos más caros.",
    "Parecía trucho, funciona mejor que otras marcas conocidas.",
    "Tenía miedo de que fuera malo, pero fue una grata sorpresa.",
    "Creí que me iban a estafar, resultó ser confiable.",
    "Me arrepentía de comprarlo, ahora pienso que fue buena idea.",
    "Al principio desconfié, terminó superando expectativas.",
    "Venía con dudas, resultó mejor de lo imaginado.",
    "Arrancó flojo, pero con el uso terminó respondiendo mejor de lo esperado.",
    "No daba dos pesos por esto y terminó siendo de lo más útil que compré.",
    "Pensé que iba a juntar polvo, pero me salvó más de una vez.",
    "La primera impresión no ayudaba, después rindió muchísimo.",
    "Creí que había comprado apurado y al final fue una buena decisión.",
]

# IRONÍA NEGATIVA SUTIL: Sarcasmo sin palabras negativas explícitas
# Estos son genuinamente difíciles porque solo tienen palabras positivas
casos_ironia_negativa = [
    "Buenísimo, ahora entiendo por qué nadie me lo había recomendado.",
    "Un lujo, sobre todo cuando tenés tiempo de sobra para esperar que responda.",
    "Qué buena compra, me vino bárbaro para aprender a leer la letra chica.",
    "Excelente, ideal si tu plan era terminar resolviendo todo por otro lado.",
    "Re práctico, salvo por el detalle de que casi nunca responde cuando lo necesitás.",
    "Una maravilla, especialmente si te gusta reiniciar todo tres veces.",
    "Perfecto, justo para esos días en los que querés complicarte solo.",
    "De primera, porque nada dice calidad como tener que improvisar desde el primer uso.",
    "Impecable, siempre deja algo nuevo para resolver.",
    "Fantástico, cada uso viene con una sorpresa distinta.",
    "Copado, me reorganizó toda la tarde aunque no era la idea.",
    "Hermoso, terminó siendo más una experiencia que una solución.",
    "Bárbaro, si lo tomás como ejercicio de paciencia.",
    "Genial, con esto aprendés rápido a bajar expectativas.",
    "Qué bien, ahora ya sé qué no volvería a elegir.",
]

# NEGACIÓN POSITIVA - Más variedad de patrones
casos_negacion_positiva = [
    "No me dio motivos para quejarme, hasta ahora respondió bien.",
    "No diría que es perfecto, pero sí bastante confiable.",
    "No esperaba acostumbrarme tan rápido, terminó rindiendo bien.",
    "No parece gran cosa, aunque en uso diario cumple mejor de lo que pensé.",
    "No fue amor a primera vista, pero después me convenció.",
    "No destaca demasiado y, sin embargo, funciona parejo.",
    "No pensé volver a usarlo, ahora quedó fijo.",
    "No lo compraría solo por la pinta, pero andando responde.",
    "No necesito decir mucho más: hace lo que promete.",
    "No me sobraban expectativas y aun así terminó sumando.",
]

# NEGACIÓN NEGATIVA - Patrones variados
casos_negacion_negativa = [
    "No diría que es un desastre, pero tampoco me resolvió nada.",
    "No se rompe ni nada raro, solo que casi nunca responde como espero.",
    "No es que no funcione nunca, pero falla justo cuando hace falta.",
    "No me dejó conforme, por más que zafe en lo básico.",
    "No esperaba maravillas, aunque igual quedó corto.",
    "No fue tan grave como para devolverlo, pero sí para no repetir.",
    "No está roto, simplemente no acompaña el uso real.",
    "No me complica siempre, aunque cuando lo necesito llega tarde.",
    "No terminó de cerrar por ningún lado.",
    "No puedo decir que sirva para lo que yo buscaba.",
]

# CASOS VERDADERAMENTE AMBIGUOS - Equilibrio perfecto de positivo/negativo
# Estos deberían confundir al modelo porque tienen IGUAL cantidad de señales
casos_ambiguos_positivos = [
    "Tiene detalles mejorables, pero en el uso diario cumple bien.",
    "No es perfecto, aunque termina resolviendo lo que necesito.",
    "Algunos aspectos flojos, pero me dejó conforme con la compra.",
    "Podría estar mejor hecho, igual lo sigo usando sin drama.",
    "Esperaba un poco más, pero por el precio acompaña.",
    "Tiene fallos menores y, aun así, la experiencia general suma.",
    "Ciertos detalles restan, aunque en conjunto lo recomendaría.",
    "No me enamoró, pero con el tiempo terminó cumpliendo.",
    "Hay cosas que corregir, sin embargo me resultó útil.",
    "No destaca tanto, aunque responde mejor de lo esperado.",
]

casos_ambiguos_negativos = [
    "Se deja usar, pero me obliga a acomodarme más yo que el producto.",
    "Tiene buen aspecto, aunque en la práctica me complica más de lo que ayuda.",
    "Cumple por momentos, pero esperaba bastante más constancia.",
    "La presentación acompaña, el rendimiento real no tanto.",
    "Lo rescatable no alcanza para tapar lo incómodo del uso.",
    "Hay cosas bien resueltas, pero el conjunto no termina de convencer.",
    "El precio no es descabellado, aunque la experiencia queda corta.",
    "Zafa para salir del paso, pero no da confianza a largo plazo.",
    "Tiene puntos a favor, aunque justo en lo importante me falló.",
    "No todo está mal, pero tampoco encuentro una razón clara para repetir.",
]

# ============================================================================
# GENERACIÓN DEL DATASET
# ============================================================================

def generar_review_positiva():
    """Genera una reseña positiva realista"""
    template = random.choice(templates_positivos)
    adj = random.choice(adjetivos_positivos)
    verbo = random.choice(verbos_positivos)
    contexto = random.choice(contextos_positivos)

    return template.format(adj=adj, verbo=verbo, contexto=contexto)

def generar_review_negativa():
    """Genera una reseña negativa realista"""
    template = random.choice(templates_negativos)
    adj = random.choice(adjetivos_negativos)
    verbo = random.choice(verbos_negativos)
    contexto = random.choice(contextos_negativos)

    return template.format(adj=adj, verbo=verbo, contexto=contexto)

# Configuración del dataset: reducimos casos simples y aumentamos difíciles
n_simples_por_clase = 250  # Reviews simples (reducido para dar más peso a casos difíciles)
n_casos_especiales = 80    # Casos difíciles (aumentado significativamente)

reviews = []
sentiments = []
tipos = []

# 1. Reviews positivas simples
for _ in range(n_simples_por_clase):
    reviews.append(generar_review_positiva())
    sentiments.append(1)
    tipos.append('simple_positivo')

# 2. Reviews negativas simples
for _ in range(n_simples_por_clase):
    reviews.append(generar_review_negativa())
    sentiments.append(0)
    tipos.append('simple_negativo')

# 3. Ironía positiva (empieza negativo, termina positivo)
for _ in range(n_casos_especiales):
    caso = random.choice(casos_ironia_positiva)
    reviews.append(caso)
    sentiments.append(1)
    tipos.append('ironia_positivo')

# 4. Ironía negativa (sarcasmo - usa palabras positivas pero es negativo)
for _ in range(n_casos_especiales):
    caso = random.choice(casos_ironia_negativa)
    reviews.append(caso)
    sentiments.append(0)
    tipos.append('ironia_negativo')

# 5. Negación positiva
for _ in range(n_casos_especiales // 2):
    caso = random.choice(casos_negacion_positiva)
    reviews.append(caso)
    sentiments.append(1)
    tipos.append('negacion_positivo')

# 6. Negación negativa
for _ in range(n_casos_especiales // 2):
    caso = random.choice(casos_negacion_negativa)
    reviews.append(caso)
    sentiments.append(0)
    tipos.append('negacion_negativo')

# 7. Casos ambiguos positivos
for _ in range(n_casos_especiales // 2):
    caso = random.choice(casos_ambiguos_positivos)
    reviews.append(caso)
    sentiments.append(1)
    tipos.append('ambiguo_positivo')

# 8. Casos ambiguos negativos
for _ in range(n_casos_especiales // 2):
    caso = random.choice(casos_ambiguos_negativos)
    reviews.append(caso)
    sentiments.append(0)
    tipos.append('ambiguo_negativo')

# Creamos DataFrame
df_full = pd.DataFrame({
    'review_body': reviews,
    'sentiment': sentiments,
    'tipo': tipos,
    'stars': [5 if s == 1 else 1 for s in sentiments],
    'review_title': [''] * len(reviews)
})

# Mezclamos aleatoriamente
df_full = df_full.sample(frac=1, random_state=42).reset_index(drop=True)

print("=" * 70)
print("DATASET GENERADO")
print("=" * 70)
print(f"\nTotal de reseñas: {len(df_full):,}")
print(f"\nDistribución por tipo:")
print(df_full['tipo'].value_counts().sort_index())
print(f"\nDistribución de sentimientos:")
print(df_full['sentiment'].value_counts())
print(f"\nBalance de clases:")
print(f"  Positivas: {(df_full['sentiment']==1).sum()/len(df_full)*100:.1f}%")
print(f"  Negativas: {(df_full['sentiment']==0).sum()/len(df_full)*100:.1f}%")

### Análisis del dataset generado

Veamos la distribución de tipos de reseñas y algunos ejemplos de cada categoría.

In [ ]:
# Trabajamos con el dataset completo
df = df_full.copy()

print("=" * 80)
print("EJEMPLOS DE RESEÑAS POR CATEGORÍA")
print("=" * 80)

# Mostramos ejemplos de cada tipo
tipos_unicos = df['tipo'].unique()

for tipo in sorted(tipos_unicos):
    print(f"\n{'='*80}")
    print(f"TIPO: {tipo.upper()}")
    print(f"{'='*80}")

    # Mostramos 3 ejemplos de este tipo
    ejemplos = df[df['tipo'] == tipo].sample(min(3, len(df[df['tipo'] == tipo])), random_state=42)

    for idx, row in ejemplos.iterrows():
        sentiment_label = "POSITIVO OK" if row['sentiment'] == 1 else "NEGATIVO X"
        print(f"\n  [{sentiment_label}] {row['review_body']}")

print(f"\n{'='*80}")
print(" NOTA PEDAGÓGICA:")
print("="*80)
print("Los casos de 'ironía' y 'negación' son DESAFIANTES para el modelo.")
print("Observá cómo palabras positivas pueden expresar sentimiento negativo")
print("(y viceversa) según el contexto. Esto es clave para entender las")
print("limitaciones de BoW/TF-IDF y motivar el uso de modelos más avanzados!")
print("="*80)

### Ejemplos específicos para la clase

Veamos algunas reseñas positivas y negativas para entender nuestros datos.

In [ ]:
# Ejemplos de reseñas positivas simples
print("=" * 80)
print("RESEÑAS POSITIVAS (SIMPLES)")
print("=" * 80)
for i, row in df[df['tipo'] == 'simple_positivo'].head(5).iterrows():
    print(f"\n★★★★★ {row['review_body']}")

# Ejemplos de reseñas negativas simples
print("\n" + "=" * 80)
print("RESEÑAS NEGATIVAS (SIMPLES)")
print("=" * 80)
for i, row in df[df['tipo'] == 'simple_negativo'].head(5).iterrows():
    print(f"\n★☆☆☆☆ {row['review_body']}")

# Casos especiales que el modelo puede fallar
print("\n" + "=" * 80)
print("CASOS DESAFIANTES (Ironía/Sarcasmo)")
print("=" * 80)
print("\n IRONÍA NEGATIVA (dice 'excelente' pero es NEGATIVO):")
for caso in casos_ironia_negativa[:3]:
    print(f"  X {caso}")

print("\n NEGACIONES COMPLEJAS:")
print(f"  OK {casos_negacion_positiva[0]}")
print(f"  X {casos_negacion_negativa[0]}")

---

## 4. División en Conjuntos de Entrenamiento y Prueba

Un principio fundamental en Machine Learning es **nunca evaluar el modelo con los mismos datos que usamos para entrenarlo**. Si lo hacemos, el modelo podría simplemente memorizar los datos (overfitting) y no generalizar bien a datos nuevos.

Por eso dividimos el dataset en dos conjuntos:
- **Entrenamiento (80%)**: Para que el modelo aprenda patrones
- **Prueba (20%)**: Para evaluar el rendimiento en datos no vistos

Además, si una reseña aparece repetida, conviene que todas sus copias queden en un solo conjunto. Si la misma oración exacta cae en entrenamiento y prueba, la evaluación deja de medir generalización real.

In [ ]:
# Hacemos el split sobre textos únicos para evitar fuga por duplicados exactos
df_unique = df.drop_duplicates(subset='review_body').copy()

df_train_unique, df_test_unique = train_test_split(
    df_unique,
    test_size=0.2,
    random_state=42,
    stratify=df_unique['sentiment']
)

train_texts = set(df_train_unique['review_body'])
test_texts = set(df_test_unique['review_body'])

df_train = df[df['review_body'].isin(train_texts)].copy()
df_test = df[df['review_body'].isin(test_texts)].copy()

reviews_train = df_train['review_body'].values
reviews_test = df_test['review_body'].values
sentiment_train = df_train['sentiment'].values
sentiment_test = df_test['sentiment'].values

print(f"Textos únicos totales: {len(df_unique):,}")
print(f"Textos únicos en entrenamiento: {len(df_train_unique):,}")
print(f"Textos únicos en prueba: {len(df_test_unique):,}")
print(f"\nTamaño del conjunto de entrenamiento: {len(reviews_train):,} reseñas")
print(f"Tamaño del conjunto de prueba: {len(reviews_test):,} reseñas")
print(f"\nDistribución en entrenamiento:")
print(f"  Positivas: {sum(sentiment_train==1):,} ({sum(sentiment_train==1)/len(sentiment_train)*100:.1f}%)")
print(f"  Negativas: {sum(sentiment_train==0):,} ({sum(sentiment_train==0)/len(sentiment_train)*100:.1f}%)")
print(f"\nChequeo rápido de fuga por textos exactos: {len(train_texts.intersection(test_texts))} coincidencias")

---

## 5. Vectorización de texto: de palabras a números

Los algoritmos de Machine Learning trabajan con números, no con texto. Necesitamos convertir las reseñas en vectores numéricos.

### Paso manual antes de automatizar

Antes de usar `CountVectorizer` o `TfidfVectorizer`, hacé esta lectura rápida:

- Tomá una reseña positiva y una negativa.
- Marcá qué palabras parecen útiles para distinguir sentimiento.
- Pensá qué se pierde si solo contamos palabras y no miramos el orden.

Esa limitación va a quedar más clara cuando comparemos estos baselines con LSTM y transformers.

### 5.1. Bag of Words con `CountVectorizer`

Esta técnica representa cada documento como un vector de conteos de palabras. Ignora el orden, pero captura frecuencia.


In [ ]:
# Creamos el vectorizador CountVectorizer
# max_features=1000 limita el vocabulario a las 1000 palabras más frecuentes
count_vectorizer = CountVectorizer(max_features=1000)

# fit() construye el vocabulario desde los datos de entrenamiento
# transform() convierte textos en matrices de conteos
X_train_counts = count_vectorizer.fit_transform(reviews_train)
X_test_counts = count_vectorizer.transform(reviews_test)

print(f"Forma de la matriz de entrenamiento: {X_train_counts.shape}")
print(f"Esto significa: {X_train_counts.shape[0]} documentos × {X_train_counts.shape[1]} palabras")
print(f"\nPrimeras 10 palabras del vocabulario: {list(count_vectorizer.get_feature_names_out()[:10])}")

### 5.2. TF-IDF (Term Frequency - Inverse Document Frequency)

TF-IDF mejora BoW al ponderar las palabras según su importancia:
- **TF (Term Frequency)**: Qué tan frecuente es una palabra en un documento
- **IDF (Inverse Document Frequency)**: Qué tan rara es esa palabra en todo el corpus

**Intuición:** Palabras como "el", "de", "la" aparecen en casi todos los documentos, por lo que tienen poco valor discriminativo. TF-IDF les asigna pesos bajos. Palabras específicas como "excelente" o "horrible" tienen pesos altos.

**Fórmula:**
```
TF-IDF(palabra, documento) = TF(palabra, documento) × IDF(palabra)
```

In [ ]:
# Creamos el vectorizador TF-IDF
tfidf_vectorizer = TfidfVectorizer(max_features=1000)

# fit_transform() combina fit() + transform()
X_train_tfidf = tfidf_vectorizer.fit_transform(reviews_train)
X_test_tfidf = tfidf_vectorizer.transform(reviews_test)

print(f"Forma de la matriz TF-IDF: {X_train_tfidf.shape}")
print(f"Tipo de matriz: {type(X_train_tfidf)} (sparse matrix para ahorrar memoria)")

---

## 6. Entrenamiento del Modelo: Regresión Logística

Vamos a entrenar dos modelos (uno con BoW y otro con TF-IDF) y comparar su rendimiento.

### ¿Por qué Regresión Logística?

Aunque el nombre dice "regresión", es un algoritmo de **clasificación**. Es simple, rápido, interpretable y funciona sorprendentemente bien como baseline para clasificación de texto.

**Ventajas:**
- Rápido de entrenar
- Requiere poca memoria
- Produce probabilidades calibradas
- Los pesos del modelo son interpretables

In [ ]:
# Modelo 1: Regresión Logística con Bag of Words
print("Entrenando modelo con Bag of Words...")
clf_bow = LogisticRegression(max_iter=1000, random_state=42)
clf_bow.fit(X_train_counts, sentiment_train)
print("OK Modelo BoW entrenado.\n")

# Modelo 2: Regresión Logística con TF-IDF
print("Entrenando modelo con TF-IDF...")
clf_tfidf = LogisticRegression(max_iter=1000, random_state=42)
clf_tfidf.fit(X_train_tfidf, sentiment_train)
print("OK Modelo TF-IDF entrenado.")

---

## 7. Evaluación de los Modelos

Evaluamos ambos modelos en el conjunto de prueba (datos que nunca vieron durante el entrenamiento).

### Métricas:

1. **Accuracy**: Porcentaje de predicciones correctas
2. **Precision**: De las reseñas que predijimos como positivas, ¿cuántas lo son realmente?
3. **Recall**: De todas las reseñas positivas reales, ¿cuántas detectamos?
4. **F1-score**: Media armónica entre precision y recall

In [ ]:
# Predicciones de ambos modelos
y_pred_bow = clf_bow.predict(X_test_counts)
y_pred_tfidf = clf_tfidf.predict(X_test_tfidf)

accuracy_bow = accuracy_score(sentiment_test, y_pred_bow)
accuracy_tfidf = accuracy_score(sentiment_test, y_pred_tfidf)

print("=" * 60)
print("RESULTADOS DE EVALUACIÓN")
print("=" * 60)
print(f"\nAccuracy con Bag of Words:  {accuracy_bow:.4f} ({accuracy_bow*100:.2f}%)")
print(f"Accuracy con TF-IDF:        {accuracy_tfidf:.4f} ({accuracy_tfidf*100:.2f}%)")
print("\n" + "=" * 60)

### Reporte de clasificación detallado (TF-IDF)

In [ ]:
print("\nREPORTE DETALLADO - MODELO TF-IDF")
print("=" * 60)
print(classification_report(sentiment_test, y_pred_tfidf,
                          target_names=['Negativo (0)', 'Positivo (1)']))

### Matriz de confusión

La matriz de confusión muestra dónde se equivoca el modelo:

```
                Predicho Neg    Predicho Pos
Real Neg             VN              FP
Real Pos             FN              VP
```

- **VP (Verdaderos Positivos)**: Correctamente clasificados como positivos
- **VN (Verdaderos Negativos)**: Correctamente clasificados como negativos
- **FP (Falsos Positivos)**: Negativos clasificados erróneamente como positivos
- **FN (Falsos Negativos)**: Positivos clasificados erróneamente como negativos

In [ ]:
cm = confusion_matrix(sentiment_test, y_pred_tfidf)

print("MATRIZ DE CONFUSIÓN - MODELO TF-IDF")
print("=" * 60)
print(f"\n{cm}\n")
print(f"Verdaderos Negativos: {cm[0,0]}")
print(f"Falsos Positivos:     {cm[0,1]}")
print(f"Falsos Negativos:     {cm[1,0]}")
print(f"Verdaderos Positivos: {cm[1,1]}")

---

## 8. Predicciones sobre Datos Nuevos en Español

Ahora que tenemos un modelo entrenado, podemos clasificar reseñas nuevas en español que nunca vio antes. Este es el objetivo final: generalizar a datos del mundo real.

In [ ]:
# Nuevas reseñas de ejemplo en español
new_reviews = [
    "Excelente producto, superó mis expectativas. Lo recomiendo totalmente.",
    "Malísima calidad, se rompió a los pocos días. No lo compren.",
    "Es aceptable, cumple su función pero nada del otro mundo.",
    "Me encantó, justo lo que buscaba. Llegó rápido y bien empaquetado.",
    "Decepcionante, no funciona como dice la descripción. Pérdida de dinero.",
    "Buenísimo, la mejor compra que podes hacer si queres tirar tu dinero a la basura.",
    "Horrible, el peor producto que compré. No sirve para nada."
]

# Vectorizamos con el MISMO vectorizador entrenado
# NUNCA usar fit() en datos nuevos! Solo transform()
X_new = tfidf_vectorizer.transform(new_reviews)

# Predicciones y probabilidades
predictions = clf_tfidf.predict(X_new)
probabilities = clf_tfidf.predict_proba(X_new)

# Mostramos resultados
print("=" * 80)
print("PREDICCIONES SOBRE RESEÑAS NUEVAS EN ESPAÑOL")
print("=" * 80)
for i, review in enumerate(new_reviews):
    sentiment_label = "POSITIVO OK" if predictions[i] == 1 else "NEGATIVO X"
    confidence = probabilities[i][predictions[i]] * 100
    print(f"\nReseña: \"{review}\"")
    print(f"Predicción: {sentiment_label} (Confianza: {confidence:.1f}%)")

---

## 9. Interpretabilidad: ¿Qué Palabras Importan?

Una ventaja de la Regresión Logística es que podemos inspeccionar los pesos del modelo para entender qué palabras en español considera más importantes para cada clase.

** Ejercicio pedagógico**: Después de ver las palabras más influyentes, analicemos por qué el modelo puede fallar en casos de ironía y sarcasmo.

In [ ]:
# Obtenemos features (palabras) y coeficientes
feature_names = tfidf_vectorizer.get_feature_names_out()
coefficients = clf_tfidf.coef_[0]

# Top 15 palabras más positivas y negativas
top_positive_indices = np.argsort(coefficients)[-15:]
top_negative_indices = np.argsort(coefficients)[:15]

print("=" * 60)
print("PALABRAS MÁS INFLUYENTES EN LAS PREDICCIONES")
print("=" * 60)

print("\nTop 15 palabras asociadas con SENTIMIENTO POSITIVO:")
for idx in reversed(top_positive_indices):
    print(f"  OK {feature_names[idx]:20s} (peso: {coefficients[idx]:.4f})")

print("\nTop 15 palabras asociadas con SENTIMIENTO NEGATIVO:")
for idx in top_negative_indices:
    print(f"  X {feature_names[idx]:20s} (peso: {coefficients[idx]:.4f})")

print("\n" + "=" * 60)
print(" ANÁLISIS PEDAGÓGICO")
print("=" * 60)
print("""
El modelo aprendió correctamente que palabras como 'excelente', 'perfecto',
'genial' están asociadas con sentimiento POSITIVO.

Sin embargo, esto es también su DEBILIDAD:

En una reseña irónica como:
  "Excelente si querés tirar la plata a la basura"

El modelo verá 'excelente' (peso positivo alto) y probablemente
la clasifique INCORRECTAMENTE como positiva, porque:

1. BoW/TF-IDF ignoran el ORDEN de las palabras
2. No capturan el CONTEXTO ("si querés tirar la plata")
3. No entienden NEGACIONES ni IRONÍA

Esto motiva el uso de modelos más avanzados (LSTM, Transformers)
que veremos en los próximos notebooks.
""")

In [ ]:
# Agregamos las predicciones al dataframe real de test
df_test = df_test.copy()
df_test['prediccion'] = y_pred_tfidf

# Calculamos accuracy por tipo de review
print("=" * 70)
print("RENDIMIENTO DEL MODELO POR TIPO DE RESEÑA")
print("=" * 70)

metricas_por_tipo = {}

for tipo in sorted(df_test['tipo'].unique()):
    df_tipo = df_test[df_test['tipo'] == tipo]
    if len(df_tipo) > 0:
        aciertos = (df_tipo['sentiment'] == df_tipo['prediccion']).sum()
        total = len(df_tipo)
        accuracy_tipo = aciertos / total * 100
        metricas_por_tipo[tipo] = accuracy_tipo

        if 'simple' in tipo:
            dificultad = "FÁCIL       "
        elif 'ironia' in tipo:
            dificultad = "DIFÍCIL     "
        else:
            dificultad = "MUY DIFÍCIL"

        print(f"\n{tipo:25s} [{dificultad}]: {accuracy_tipo:5.1f}% ({aciertos}/{total})")

familias = {}
for tipo, accuracy_tipo in metricas_por_tipo.items():
    familia = tipo.split('_')[0]
    familias.setdefault(familia, []).append(accuracy_tipo)

accuracy_por_familia = {familia: np.mean(valores) for familia, valores in familias.items()}
peor_tipo = min(metricas_por_tipo, key=metricas_por_tipo.get)
simple_acc = accuracy_por_familia.get('simple')
familias_complejas = [familia for familia in ['ironia', 'negacion', 'ambiguo'] if familia in accuracy_por_familia]
brecha_clara = simple_acc is not None and any(simple_acc - accuracy_por_familia[familia] >= 10 for familia in familias_complejas)

print("\n" + "=" * 70)
print("OBSERVACIONES PEDAGÓGICAS")
print("=" * 70)
print("\nPromedio de accuracy por familia:")
for familia in ['simple', 'ironia', 'negacion', 'ambiguo']:
    if familia in accuracy_por_familia:
        print(f"  - {familia:8s}: {accuracy_por_familia[familia]:5.1f}%")

print(f"\nPeor tipo observado: {peor_tipo} ({metricas_por_tipo[peor_tipo]:.1f}%)")

if brecha_clara:
    print("\nEn esta corrida, los casos simples quedaron claramente por encima de los casos con mayor dependencia contextual.")
    print("Eso es consistente con la intuición del notebook: TF-IDF capta bien palabras sueltas, pero se vuelve más frágil con ironía, contraste y alcance de negaciones.")
else:
    print("\nEn esta corrida no apareció una brecha grande entre casos simples y complejos.")
    print("Eso no prueba comprensión profunda: en un dataset sintético todavía puede haber pistas léxicas fuertes o pocos patrones realmente nuevos en test.")
    print("La lectura correcta es que este baseline sirve como referencia, pero sus resultados deben interpretarse con cautela.")

print("\nCONCLUSIÓN: TF-IDF + Logistic Regression sigue siendo un buen baseline interpretable,")
print("pero cuando el significado depende mucho del contexto conviene no sobreinterpretar un accuracy alto.")

### Análisis de Rendimiento por Tipo de Reseña

Veamos cómo le va al modelo en diferentes tipos de casos.

---

##  Guía Teórico-Conceptual

### 1. Flujo completo de un proyecto de clasificación de texto

**Paso 1: Recolección de datos**  
Obtener un corpus etiquetado (en nuestro caso, reseñas con sentimiento)

**Paso 2: Preprocesamiento**  
Limpiar texto, tokenización, opcional: stemming/lemmatización

**Paso 3: Vectorización**  
Convertir texto en representación numérica (BoW, TF-IDF, embeddings)

**Paso 4: División train/test**  
Separar datos para entrenar y evaluar sin sesgo

**Paso 5: Entrenamiento**  
Ajustar modelo a los datos de entrenamiento

**Paso 6: Evaluación**  
Medir rendimiento en datos de prueba

**Paso 7: Optimización**  
Ajustar hiperparámetros, probar otros modelos

**Paso 8: Despliegue**  
Usar el modelo en producción

---

### 2. Bag of Words vs. TF-IDF

**Bag of Words (BoW):**
- Representa documentos como vectores de conteos de palabras
- Ignora orden y gramática
- Simple pero efectivo como baseline
- **Problema**: Palabras muy frecuentes dominan la representación

**TF-IDF:**
- Balancea frecuencia local (documento) con rareza global (corpus)
- Palabras comunes reciben pesos bajos
- Palabras discriminativas reciben pesos altos
- Generalmente supera a BoW en clasificación de texto

---

### 3. Regresión Logística para Clasificación

**Funcionamiento:**
- Aprende función lineal: z = w₁x₁ + w₂x₂ + ... + wₙxₙ + b
- Aplica sigmoide: P(y=1|x) = 1 / (1 + e⁻ᶻ)
- Salida es probabilidad entre 0 y 1
- Si P > 0.5 → Clase 1, sino → Clase 0

**Ventajas:**
- Rápido y eficiente
- Probabilidades calibradas
- Pesos interpretables
- Funciona bien con alta dimensionalidad

**Limitaciones:**
- Asume relaciones lineales
- No captura interacciones complejas
- Ignora orden de palabras

---

### 4. Métricas de Evaluación

**Accuracy:** Porcentaje de predicciones correctas (cuidado con clases desbalanceadas)

**Precision:** De las predichas positivas, ¿cuántas son realmente positivas?

**Recall:** De las positivas reales, ¿cuántas detectamos?

**F1-Score:** Media armónica entre precision y recall

---

### 5. Importancia del Baseline

Antes de usar redes neuronales complejas, siempre establecemos un baseline simple para:

1. Entender la dificultad del problema
2. Detectar problemas en los datos
3. Tener punto de comparación cuantitativo
4. Justificar complejidad adicional
5. Iterar rápidamente

En muchos casos, un modelo simple bien ajustado es suficiente y preferible (más rápido, interpretable, fácil de mantener).

---

##  Preguntas y Respuestas para Estudio

### Preguntas Conceptuales

**1. ¿Por qué es importante dividir los datos en conjuntos de entrenamiento y prueba?**

*Respuesta:* Para evaluar el rendimiento del modelo en datos que nunca vio durante el entrenamiento. Si evaluáramos con los mismos datos de entrenamiento, el modelo podría haber memorizado los ejemplos (overfitting) y no generalizar bien a datos nuevos del mundo real.

---

**2. ¿Cuál es la diferencia principal entre Bag of Words y TF-IDF?**

*Respuesta:* BoW solo cuenta la frecuencia de cada palabra en el documento, mientras que TF-IDF pondera esa frecuencia según qué tan rara es la palabra en todo el corpus. TF-IDF reduce la importancia de palabras muy comunes (como "el", "de") y aumenta la de palabras discriminativas.

---

**3. ¿Por qué usamos Regresión Logística para clasificación si su nombre dice "regresión"?**

*Respuesta:* Aunque el nombre es confuso, la Regresión Logística es un algoritmo de clasificación. Usa una función logística (sigmoide) para convertir una combinación lineal de features en una probabilidad entre 0 y 1, y luego clasifica según un umbral (típicamente 0.5).

---

**4. ¿Qué es el overfitting y cómo lo evitamos en este notebook?**

*Respuesta:* Overfitting ocurre cuando el modelo memoriza los datos de entrenamiento en lugar de aprender patrones generalizables. Lo evitamos mediante: (1) división train/test, (2) limitación del vocabulario (max_features=1000), y (3) regularización implícita en LogisticRegression.

---

**5. ¿Por qué nunca debemos usar fit() en los datos de prueba?**

*Respuesta:* Porque fit() aprende parámetros de los datos (vocabulario, escalas, etc.). Si lo usamos en datos de prueba, el modelo "espía" información que no debería conocer, invalidando la evaluación. Solo debemos usar transform() en datos nuevos.

---

### Preguntas Técnicas

**6. Si tenemos un dataset con 90% de reseñas positivas y 10% negativas, ¿qué problema tiene usar solo accuracy como métrica?**

*Respuesta:* Un modelo trivial que prediga "positivo" para todo tendría 90% de accuracy sin aprender nada útil. En datasets desbalanceados, debemos usar precision, recall y F1-score para evaluar el rendimiento en cada clase.

---

**7. ¿Qué significa que TfidfVectorizer devuelva una "matriz dispersa" (sparse matrix)?**

*Respuesta:* Como la mayoría de las entradas son cero (cada documento solo contiene una pequeña fracción del vocabulario total), scipy usa una representación dispersa que solo almacena los valores no-cero. Esto ahorra memoria y acelera cálculos.

---

**8. ¿Cómo interpretamos los coeficientes de la Regresión Logística?**

*Respuesta:* Coeficientes positivos indican que la presencia de esa palabra aumenta la probabilidad de clase positiva. Coeficientes negativos indican asociación con la clase negativa. La magnitud indica la fuerza de la asociación.

---

**9. ¿Qué pasaría si no usáramos random_state en train_test_split?**

*Respuesta:* Cada ejecución del notebook produciría una división diferente, resultando en métricas ligeramente distintas. Fijar random_state garantiza reproducibilidad, importante para debugging y comparación de modelos.

---

**10. ¿Por qué limitamos max_features a 1000 palabras?**

*Respuesta:* Para reducir dimensionalidad y evitar overfitting. Palabras muy raras (que aparecen en 1-2 documentos) suelen ser ruido o typos. Mantener solo las más frecuentes captura la mayor parte de la información con menor riesgo de sobreajuste.

---

### Preguntas de Aplicación

**11. Mencioná tres limitaciones del enfoque BoW/TF-IDF que las redes neuronales podrían superar.**

*Respuesta:*
1. Ignoran el orden de las palabras ("no es bueno" vs "es bueno")
2. No capturan significado semántico ("excelente" y "genial" son tratadas como completamente diferentes)
3. No modelan dependencias largas ni contexto complejo

---

**12. Si tuvieras que clasificar tweets (textos muy cortos), ¿qué ajustes harías a este enfoque?**

*Respuesta:*
- Incluir n-gramas (bigramas, trigramas) para capturar frases cortas
- Reducir max_features (menos palabras únicas en tweets)
- Considerar preprocesamiento de hashtags, menciones y emojis
- Probar con char-level features para manejar jerga y typos

---

**13. ¿En qué casos preferirías usar Regresión Logística sobre una red neuronal profunda?**

*Respuesta:*
- Dataset pequeño (pocas muestras)
- Necesidad de interpretabilidad (explicar decisiones)
- Recursos computacionales limitados
- Necesidad de entrenar/desplegar rápidamente
- Cuando el baseline ya da resultados satisfactorios

---

##  Ejercicios Propuestos

### Ejercicio 1: Experimentación con Hiperparámetros
Probá cambiar `max_features` en TfidfVectorizer a 500, 2000 y 5000. ¿Cómo afecta al accuracy? ¿Observás overfitting con vocabularios muy grandes?

### Ejercicio 2: N-gramas
Modificá TfidfVectorizer para incluir bigramas: `TfidfVectorizer(max_features=1000, ngram_range=(1,2))`. ¿Mejora el rendimiento? ¿Por qué los bigramas pueden ser útiles en español?

### Ejercicio 3: Dataset Completo
Probá entrenar con el dataset completo (200,000 reviews) en lugar del subset de 10,000. ¿Mejora significativamente el accuracy? ¿Cuánto tarda el entrenamiento?

### Ejercicio 4: Análisis de Errores
Identificá 5 reseñas del conjunto de prueba que el modelo clasificó incorrectamente. ¿Qué tienen en común? ¿Son casos difíciles incluso para humanos? (Tip: ironía, sarcasmo, negaciones)

### Ejercicio 5: Comparación de Modelos
Probá otros clasificadores de sklearn: `MultinomialNB`, `SVC`, `RandomForestClassifier`. ¿Cuál da mejores resultados en español? ¿Cuál es más rápido?

### Ejercicio 6: Dataset Propio
Buscá otro dataset en español (Twitter, reviews de apps, noticias) y aplicá el mismo pipeline. ¿El modelo generaliza bien a otros dominios?

---

## Conclusión

En este notebook construimos un baseline clásico para clasificación de sentimientos en español. El recorrido dejó cuatro ideas importantes:

1. Un modelo simple puede rendir muy bien si la representación del texto está bien elegida.
2. TF-IDF suele ofrecer una señal más útil que Bag of Words puro cuando hay palabras muy frecuentes.
3. La evaluación sobre un conjunto de prueba separado es obligatoria si queremos hablar de generalización.
4. Los casos de ironía, sarcasmo y negación muestran una limitación importante: cuando el sentido depende mucho del contexto, estos modelos pueden volverse frágiles o dar una falsa sensación de seguridad.

## Qué sigue

En el próximo notebook vamos a mantener la tarea de clasificación, pero cambiaremos el foco: veremos Naive Bayes y pipelines para organizar mejor el flujo de trabajo y preparar la transición hacia modelos neuronales.
